In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import current_timestamp, current_date
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "proyecto_olist")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsssmartdataanayeli2")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/olist/orders/"
checkpoint_path = f"abfss://{container}@{storageName}.dfs.core.windows.net/_checkpoints/orders/"

In [0]:
# Lectura incremental
df_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("cloudFiles.schemaLocation", checkpoint_path)
  .load(ruta))

In [0]:
df_bronze = (
    df_stream
    .withColumn(
        "order_purchase_timestamp",
        F.to_timestamp("order_purchase_timestamp")
    )
    .withColumn(
        "order_approved_at",
        F.to_timestamp("order_approved_at")
    )
    .withColumn(
        "order_delivered_carrier_date",
        F.to_timestamp("order_delivered_carrier_date")
    )
    .withColumn(
        "order_delivered_customer_date",
        F.to_timestamp("order_delivered_customer_date")
    )
    .withColumn(
        "order_estimated_delivery_date",
        F.to_timestamp("order_estimated_delivery_date")
    )
    .drop("_rescued_data")
    .withColumn("ingestion_date", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("processing_date", F.current_date())
)

In [0]:
(df_bronze.writeStream
  .format("delta")
  .outputMode("append")
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .toTable("proyecto_olist.bronze.orders"))